In [7]:
import pandas as pd, numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

result = pd.read_excel(r"..\data\public_data_resultsF.xlsx")
surv   = pd.read_excel(r"..\data\private_dataF.xlsx")

## official counts

In [12]:
# total sum of 1060
result = result.rename(columns={result.columns[0]: "group", "Invalid ballots": "Invalid"}).set_index("group")
cats = ["Red", "Green", "Invalid"]
off = result[cats]
off_e = off.loc["E-votes"]
off_p = off.loc[[i for i in off.index if i.startswith("Polling")]]
off_paper = off_p.sum()
pd.concat([off_e.to_frame("E-votes").T, off_paper.to_frame("Paper voters").T])

,Red,Green,Invalid
E-votes,111,221,10
Paper voters,242,466,10


## survey counts

In [ ]:
# total sum of 200 votes
surv["party"] = surv["party"].replace({"Invalid vote": "Invalid"})
obs_e = surv[surv.evote == 1]["party"].value_counts().reindex(cats, fill_value=0)
obs_p = surv[surv.evote == 0]["party"].value_counts().reindex(cats, fill_value=0)
pd.DataFrame({"E-voters": obs_e, "Paper voters": obs_p}).T

party,Red,Green,Invalid
E-voters,21,37,4
Paper voters,46,90,2


In [ ]:
# small nr of invalid votes, will analyse only green and red parties
# degrees_of_freedom=1 since we analyse 2 parties, df=K-1
def gof(name, obs, official, cats):
    o = obs[cats]
    p = official[cats] / official[cats].sum()
    exp = p * o.sum()
    chi = stats.chisquare(o, exp)
    print(f"{name}: chi2={chi.statistic:.3f}, p={chi.pvalue:.4f}")
    print(pd.DataFrame({"observed": o, "expected": exp.round(1)}))

gof("E-voters",     obs_e,      off_e,      ["Red", "Green"])
print("****************************")
gof("Paper voters", obs_p,      off_paper,  ["Red", "Green"])

#large p-values (way over p=0.05), cannot reject H0, no evidence survey differs from the results
#low χ² measures how far the observed survey counts are from expectations under the official results -  0 perfect  match, larger values are mismatch

E-voters: chi2=0.200, p=0.6544
       observed  expected
Red          21      19.4
Green        37      38.6
****************************
Paper voters: chi2=0.008, p=0.9300
       observed  expected
Red          46      46.5
Green        90      89.5
